In [8]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ"
    r"\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist"
    r"\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"
)

In [9]:
from google.cloud import bigquery
from google.oauth2 import service_account

# Google Cloud credentials
credentials = service_account.Credentials.from_service_account_file(

    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"

)

# Client
client = bigquery.Client(project="gen-lang-client-0137385761", credentials=credentials)

In [10]:
# Construct a reference to the "hacker_news" dataset
dataset_ref = client.dataset("hacker_news", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

# List all tables in the dataset
tables = list(client.list_tables(dataset))
print(f"Tables in 'hacker_news' dataset: {len(tables)}")
for table in tables:
    print(f"  - {table.table_id}")

Tables in 'hacker_news' dataset: 1
  - full


In [11]:
# Construct a reference to the "full" table
table_ref = dataset_ref.table("full")

# API request - fetch the table
table = client.get_table(table_ref)

# View all columns: name, data type, and description
print(f"Table: {table.full_table_id}")
print(f"Rows: {table.num_rows:,}  |  Columns: {len(table.schema)}\n")
print(f"{'Column':<20} {'Type':<15} Description")
print("-" * 70)
for field in table.schema:
    print(f"{field.name:<20} {field.field_type:<15} {field.description or ''}")

Table: bigquery-public-data:hacker_news.full
Rows: 48,349,033  |  Columns: 14

Column               Type            Description
----------------------------------------------------------------------
title                STRING          Story title
url                  STRING          Story url
text                 STRING          Story or comment text
dead                 BOOLEAN         Is dead?
by                   STRING          The username of the item's author.
score                INTEGER         Story score
time                 INTEGER         Unix time
timestamp            TIMESTAMP       Timestamp for the unix time
type                 STRING          type of details (comment comment_ranking poll story job pollopt)
id                   INTEGER         The item's unique id.
parent               INTEGER         Parent comment ID
descendants          INTEGER         Number of story or poll descendants
ranking              INTEGER         Comment ranking
deleted              BOOL

In [12]:
# Preview the first five rows of the table
client.list_rows(table, max_results=5).to_dataframe(create_bqstorage_client=False)

,title,url,text,dead,by,score,time,timestamp,type,id,parent,descendants,ranking,deleted
0,NaN,NaN,NaN,<NA>,NaN,<NA>,1437517899,2015-07-21 22:31:39+00:00,story,9926338,<NA>,<NA>,<NA>,<NA>
1,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaT,story,9926520,<NA>,<NA>,<NA>,<NA>
2,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaT,story,9926526,<NA>,<NA>,<NA>,<NA>
3,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaT,story,9926529,<NA>,<NA>,<NA>,<NA>
4,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaT,story,9926535,<NA>,<NA>,<NA>,<NA>


In [13]:
# Preview the first five rows of the table
query = """
    SELECT *
    FROM `bigquery-public-data.hacker_news.full`
    WHERE `by` IS NOT NULL
      AND title IS NOT NULL
    LIMIT 5
"""
client.query(query).to_dataframe(create_bqstorage_client=False)

,title,url,text,dead,by,score,time,timestamp,type,id,parent,descendants,ranking,deleted
0,Placeholder,NaN,Mind the gap.,<NA>,kogir,0,1401561473,2014-05-31 18:37:53+00:00,story,994369,<NA>,0,<NA>,<NA>
1,"Dharma – A generation-based, context-free gram...",http://www.kitploit.com/2015/07/dharma-generat...,NaN,True,Dump3R,1,1437516986,2015-07-21 22:16:26+00:00,story,9926258,<NA>,<NA>,<NA>,<NA>
2,Seven Digital Deadly Sins: An Interactive Refl...,http://sins.nfb.ca/,NaN,<NA>,32faction,1,1437517026,2015-07-21 22:17:06+00:00,story,9926260,<NA>,0,<NA>,<NA>
3,Hackers Remotely Kill a Jeep on the Highway–Wi...,http://www.wired.com/2015/07/hackers-remotely-...,NaN,True,bloat,1,1437517060,2015-07-21 22:17:40+00:00,story,9926263,<NA>,<NA>,<NA>,<NA>
4,What is code?,http://www.bloomberg.com/graphics/2015-paul-fo...,NaN,True,grey-area,1,1437517070,2015-07-21 22:17:50+00:00,story,9926264,<NA>,<NA>,<NA>,<NA>


In [18]:
# Query to select prolific commenters and post counts over 10,000
prolific_commenters_query = """
        SELECT `by`, COUNT(1) AS NumPosts
        FROM `bigquery-public-data.hacker_news.full`
        GROUP BY `by`
        HAVING COUNT(1) > 10000
        """

# Set up the query (cancel the query if it would use too much of 
# your quota, with the  limit set to 1 GB)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)
query_job = client.query(prolific_commenters_query, job_config=safe_config)

# API request - run the query, and return a pandas DataFrame
prolific_commenters = query_job.to_dataframe(create_bqstorage_client=False)

# View top few rows of results
print(prolific_commenters.head())

           by  NumPosts
0   agumonkey     21562
1  Brajeshwar     15666
2  prostoalex     11506
3     bombcar     20779
4    jacquesm     65646


In [25]:
# How many comments have been deleted?
query = """
        SELECT COUNT(1) AS NumDeletedPosts
        FROM `bigquery-public-data.hacker_news.full`
        WHERE deleted = TRUE
        """

# Set up the query (cancel the query if it would use too much of 
# your quota, with the limit set to 1 GB)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)
query_job = client.query(query, job_config=safe_config)

# API request - run the query, and return a pandas DataFrame
num_deleted_posts_df = query_job.to_dataframe(create_bqstorage_client=False)

# Extract the count as a number (what the grader checks)
num_deleted_posts = num_deleted_posts_df['NumDeletedPosts'][0]

print(num_deleted_posts)

0
